In [ ]:
import base64
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Image, HTML

In [ ]:
load_dotenv("../.env") # load our env vars
client = OpenAI(
    base_url="https://api.meta.ai/v1",
    api_key=os.environ["MODEL_API_KEY"],
)

In [ ]:
# Helper to display output images
def display_image_table(images, mime_type="image/webp", width=220):
    html = "<table><tr>"

    for i, image_b64 in enumerate(images):
        html += f"""
        <td style="padding: 8px; text-align: center; vertical-align: top;">
            <img src="data:{mime_type};base64,{image_b64}"
                 style="width: {width}px; height: auto; border-radius: 8px;">
            <div style="margin-top: 6px;">Image {i + 1}</div>
        </td>
        """
        if (i + 1) % 3 == 0:
            html += "</tr><tr>"

    html += "</tr></table>"
    display(HTML(html))

# Helper to convert local files into Responses API image input items 
def make_input_image_item(path):
    path = Path(path)
    ext = path.suffix.lower()

    mime_map = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".webp": "image/webp",
    }

    mime_type = mime_map.get(ext, "image/png")
    image_b64 = base64.b64encode(path.read_bytes()).decode("utf-8")

    return {
        "type": "input_image",
        "image_url": f"data:{mime_type};base64,{image_b64}",
    }    

def extract_images(response):
    return [
        item.result
        for item in response.output
        if item.type == "image_generation_call"
    ]

In [ ]:
# WITHOUT generate_turn() — manually managing each conversation turn.
#
# turn1 = client.responses.create(
#     model="muse-image-1.0",
#     input="Create a luxury canned meat brand called Oinker's."
# )
# turn2 = client.responses.create(
#     model="muse-image-1.0",
#     previous_response_id=turn1.id,
#     input="Make the packaging navy blue with elegant gold lettering."
# )
# turn3 = client.responses.create(
#     model="muse-image-1.0",
#     previous_response_id=turn2.id,
#     input=[{"role": "user",
#           "content": [
#               {"type": "input_text", "text": "Add the pig mascot from this reference image." },
#               make_input_image_item("oinkers.png")
#           ]
#      }]
# )
def generate_turn(prompt, image_paths=None, prev_turn=None):
    content = [
        {"type": "input_text", "text": prompt},
        *[make_input_image_item(path) for path in (image_paths or [])]
    ]
    params = {
        "model": "muse-image-1.0",
        "input": [{"role": "user", "content": content}],
        "tools": [{"type": "image_generation","size": "1024x1024"}
    ]
    }
    if prev_turn is not None:
        params["previous_response_id"] = prev_turn["response"].id
    response = client.responses.create(**params)
    return {
        "response": response,
        "images": extract_images(response)
    }

In [ ]:
# A major difference between the one-off-generation and conversation generation
# is the /v1/responses API does not allow you to return multiple images in a single turn.
# where the one-off-generation has a n parameter.

turn = generate_turn(
    prompt="""
Create a canned lunch meat brand called Oinker's.

Show a classic SPAM-style luncheon meat can:
a short, squat, compact rectangular metal can with heavily rounded corners,
wider than it is tall,
with a shallow depth,
flat top and bottom,
and a wrapped paper label around the sides.

It should look like a small block-shaped luncheon meat can,
not a tall vertical tin,
not a cylindrical soup can,
and not a tray.

Show it in a clean product-photo style, 3/4 view, so both the front face and one side are visible.
"""
)
display_image_table(turn["images"])

In [ ]:
turn = generate_turn(prompt="Make the packaging navy blue with elegant gold lettering..",prev_turn=turn)
display_image_table(turn["images"])

In [ ]:
turn = generate_turn(
    prompt="Use the pig mascot from this reference image. Take up half the labelling",
    image_paths=["oinkers.png"],
    prev_turn=turn
)

display_image_table(turn["images"])